**About this copy.** This is the MNIST copy of `model_fitting/4_nonlinear_exploration.ipynb` on the `zain` branch. It reads the tables in `mnist/sweep/` that `mnist_sweep.py assemble` writes, through `mnist_settings.py`. The methods and the order of the parts are the same as in the original. These things changed:
- The knob `alpha` is replaced by the initial scale s (column `init_scale`). Every held-out `alpha` evaluation is now a held-out s evaluation.
- The grokking step `T` is the first saved step with test accuracy at least 0.80, where the original used 0.95.
- The prediction time $t^*$ is `M.LANDMARK`, the kernel step nearest `M.LANDMARK_TARGET`. The original fixed it at step 2,430.
- The original kept weight decay up to 1e-3. Here a weight decay level is dropped when no run at that level memorises (training accuracy at least 0.99). The Load cell prints which levels were dropped.
- The grid values, the step budget and the kernel steps are read from the data, so the notebook runs on any part of the grid.
- Every statement of a result from modular addition was removed. The places where a result belongs say so.

Set the environment variable `MNIST_SWEEP_DATA` to read the tables from another folder.

# Notebook 4. The shape of the relation, and how much is noise

**Question.** Among runs that grok, how does the grokking step depend on the kernel statistics `A_t`, `S_t` and `R_t`? Is the relation a straight line, what shape does it take, do the statistics interact, and how much of the data is noise?

Notebook 3 used all runs, including runs that never grok, and asked whether the statistics rank runs from a new initial scale s. Here the main tool is kernel ridge regression, which fits a smooth surface with no shape fixed in advance. We compare it with simpler models to see what each kind of shape adds.

An interaction term here is the product of two standardised statistics: $z_A z_S$, $z_A z_R$ or $z_S z_R$. It lets the effect of one statistic depend on another. As in notebook 3, the models use log `R_t` (Part 1 explains why).

**Short answer.** To be written once the MNIST sweep has run.

## Words used in this notebook

| Term | Meaning |
| --- | --- |
| grokking step `T` | The first saved step at which test accuracy reaches 0.80. The golden report (§3.5) uses 0.95 on modular addition; MNIST does not reach 0.95 reliably. |
| initial scale s | The factor that multiplies the initial weights (column `init_scale`). It takes the place of `alpha` in the modular addition notebooks. |
| cell | One mix of s, width and weight decay. Each cell has 5 seeds. |
| `S_t`, `R_t`, `A_t` | Kernel scale, rotation and alignment at step t (golden report Eq. 6). `S_t` is the kernel's size relative to the start (1 means no change). `R_t` is one minus the cosine between the kernel now and at the start (0 means the same shape). `A_t` is the alignment with the labels of a fixed draw of 1,000 test images. |
| prediction time $t^*$ | The kernel step `M.LANDMARK`, which must come before every run groks. The statistics are read here. |
| cross-sectional covariates | The three statistics at $t^*$. Parts 1 to 6 use these. |
| longitudinal covariates | The statistics at all kernel steps up to $t^*$. Part 7 uses them. |
| standardised | Minus the mean and divided by the standard deviation of the training runs. We write $z_A$, $z_S$, $z_R$. |
| between-cell and within-cell spread | The spread of a quantity splits in two: how much the cell averages differ (set by the settings), and how much the seeds of one cell differ from their average (seed noise). |
| $R^2$ | The share of the spread in $\log T$ that a model explains on held-out runs. 1 is perfect, 0 is no better than the mean, and below 0 is worse than the mean. |
| concordance | Pick two runs. It is the share of pairs that the model puts in the right order. 0.5 is a coin toss and 1 is perfect. |
| additive model | $\log T = \beta_0 + f_A(z_A) + f_S(z_S) + f_R(z_R)$. Each statistic has its own curve, and none changes the effect of another. |
| interaction term | The product of two standardised statistics: $z_A z_S$, $z_A z_R$ or $z_S z_R$. With the term $\gamma_{AS} z_A z_S$, the slope of $\log T$ in $z_A$ changes by $\gamma_{AS}$ for each standard deviation of $z_S$. |
| partial dependence | The model's average prediction as one statistic slides from low to high, with the other two left at each run's own values. |

The data are runs of a bias-free 3-layer ReLU network trained on 1,000 MNIST training images and tested on all 10,000 test images, each for a budget of 500,000 steps.

## How the models are tested

**Which runs.** Parts 1 to 5 and Part 7 use the runs that grokked, because their $\log T$ is a known number. The Load cell prints how many there are for each s. This selects the faster runs. So these parts answer "when does a run grok, given that it groks?". Part 6 uses all runs.

**No random train/test split of runs.** The 5 seeds of a cell are near copies. A random split would put copies on both sides and reward remembering cells. Every score holds out a whole group instead.
- **Hold out one cell.** One fold per cell. The model has seen cells with nearby settings, so this tests the shape of the fit inside the range of the data.
- **Hold out one s.** One fold per value of s, as in notebook 3. This tests a value of s the model has never seen. We report the concordance on each held-out s and the mean over them.

Fits on all grokked runs, such as the curves of Part 4, describe the data and make no claim about prediction.

**Choosing model settings.** Settings are always chosen inside the training runs of a fold.
- The ridge penalty of the linear-with-penalty and spline models is chosen from 13 values by leave-one-run-out cross-validation (`RidgeCV`). That leaves out single runs, while the other seeds of the cell stay in.
- Kernel ridge chooses its penalty and width by 5-fold cross-validation that holds out whole cells.
- XGBoost in Part 7 holds out 20% of the training cells as a validation set, tries 24 settings with early stopping, and keeps the one with the lowest validation error.
- Part 6 fixes every setting in advance, so the two models in each pair differ only in whether the statistics may interact.

These notebooks are exploratory and compare many models, so treat small gaps (below about 0.02 in $R^2$ or 0.03 in concordance) as noise.

## Setup

In [ ]:
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from joblib import Parallel, delayed
from lifelines import LogNormalAFTFitter
from lifelines.utils import concordance_index
from sklearn.compose import TransformedTargetRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler

import mnist_settings as M

plt.style.use('fitting.mplstyle')
GREY, INK, PALE = '#8a8984', '#0b0b0b', '#c8c7c2'
KERNEL_COLOURS = {'A_t': '#7b5cd6', 'S_t': '#1a9e8f', 'R_t': '#c99a06', 'log R_t': '#c99a06'}
S_PALETTE = ['#2a78d6', '#0b0b0b', '#eb6834', '#c2307a', '#5e9e2a', '#8a5a2b']   # one colour per initial scale s, smallest first
KNOB = M.KNOB                                                           # the initial scale s, column init_scale
kernel = ['A_t', 'S_t', 'R_t']                                          # the statistics as stored
covs = ['A_t', 'S_t', 'log R_t']                                        # what the models use (Part 1 explains log R_t)
cov_pairs = [('A_t', 'S_t'), ('A_t', 'log R_t'), ('S_t', 'log R_t')]

## Load the data

`M.load()` returns the runs, the accuracy at every checkpoint, the kernel statistics at about 48 log-spaced kernel steps, and the prediction time $t^*$. A weight decay level is dropped when no run at that level memorises.

In [ ]:
runs, acc, kern, T_STAR = M.load()                                      # T_STAR is the prediction time t*
print(f'Prediction time t* = step {T_STAR:,}. Data read from {M.DATA}.')

events = runs.set_index('run_id')
events['T'] = acc[acc['test_acc'] >= M.LEVEL].groupby('run_id')['step'].min()
events['memorised'] = acc.groupby('run_id')['train_acc'].max() >= M.MEM
events = events.reset_index()
events['cell'] = events.apply(M.cell_name, axis=1)

# Drop the weight decay levels at which no run memorises. The original kept eta_kappa <= 1e-3 instead.
memorising = events.groupby('eta_kappa')['memorised'].any()
dropped_decays = sorted(memorising.index[~memorising])
print('Weight decay levels dropped because no run memorises:', [f'{d:g}' for d in dropped_decays] or 'none')

everything = events.merge(kern[kern['step'] == T_STAR].drop(columns='step'), on='run_id')
everything = everything[~everything['eta_kappa'].isin(dropped_decays)].reset_index(drop=True)
assert (everything['R_t'] > 0).all(), 'R_t is 0 for some run at t*, so log R_t is undefined. Choose a later M.LANDMARK_TARGET.'
everything['log R_t'] = np.log(everything['R_t'])
data = everything.dropna(subset=['T']).reset_index(drop=True)     # grokked runs only
log_T = np.log(data['T'])
assert data['T'].min() > T_STAR, (f'A run reaches test accuracy {M.LEVEL} at step {data["T"].min():,}, '
                                  f'before the kernel is read at t* = {T_STAR:,}. Lower M.LANDMARK_TARGET in mnist_settings.py.')

S_VALUES = sorted(everything[KNOB].unique())
S_COLOURS = dict(zip(S_VALUES, S_PALETTE))
BUDGET = int(everything['last_step'].max())
print(f'{len(everything)} runs in {everything["cell"].nunique()} cells, of which {len(data)} grokked. Step budget {BUDGET:,}.')
pd.DataFrame({'runs': everything.groupby(KNOB).size(), 'grokked runs': data.groupby(KNOB).size()}).fillna(0).astype(int).rename_axis('s')

## Part 1. Look at the data

**Question.** What does each statistic look like against the grokking step?

Each dot is a grokked run, coloured by s. A straight-line relation would show as a tilted band. `R_t` is drawn on a log axis because it can span several orders of magnitude.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, stat in zip(axes, kernel):
    for s, g in data.groupby(KNOB):
        ax.scatter(g[stat], g['T'], s=12, color=S_COLOURS[s], alpha=0.6, label=f's = {s:g}')
    ax.set_yscale('log')
    ax.set_xlabel(stat)
axes[2].set_xscale('log')
axes[2].set_xlabel('R_t (log scale)')
axes[0].set_ylabel('Grokking step T')
axes[0].legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 2. How much is signal and how much is noise?

**Question.** The 5 seeds of a cell share every setting. How much of the spread in $\log T$ and in each statistic comes from the settings, and how much from the seed?

For each quantity we split its spread in two. The between-cell part is how much the cell averages differ. The within-cell part is how much seeds differ from their cell average, which is seed noise. We also ask how much a single setting explains when it is used on its own.

The saved checkpoints also round `T` up to the next saved step, which adds a small measurement error. The cell prints its size.

In [ ]:
values = {'log T': log_T, 'A_t': data['A_t'], 'S_t': data['S_t'], 'log R_t': data['log R_t']}
groupings = {'s only': KNOB, 'width only': 'width', 'weight decay only': 'eta_kappa', 'all three (the cell)': 'cell'}
explained = pd.DataFrame({name: {label: 1 - ((v - v.groupby(data[key]).transform('mean')) ** 2).sum() / ((v - v.mean()) ** 2).sum()
                                 for label, key in groupings.items()} for name, v in values.items()}).T
explained['seed noise'] = 1 - explained['all three (the cell)']

steps = np.sort(acc['step'].unique())
previous = steps[np.searchsorted(steps, data['T']) - 1]     # T lies somewhere after the previous saved step
rounding = ((log_T - np.log(previous)) ** 2 / 12).mean() / log_T.var()
print(f'Rounding T up to a saved step adds about {rounding:.2%} of the spread of log T.')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), gridspec_kw={'width_ratios': [1, 1.4]})
axes[0].barh(explained.index, explained['all three (the cell)'], color=INK, label='between cells (settings)')
axes[0].barh(explained.index, explained['seed noise'], left=explained['all three (the cell)'], color=PALE, label='within cells (seed noise)')
axes[0].set_xlabel('Share of the spread')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=8)
axes[0].invert_yaxis()
width = 0.25
for j, (label, colour) in enumerate(zip(['s only', 'width only', 'weight decay only'], [S_COLOURS[S_VALUES[-1]], KERNEL_COLOURS['A_t'], KERNEL_COLOURS['S_t']])):
    axes[1].barh(np.arange(len(explained)) + (j - 1) * width, explained[label], height=width, color=colour, label=label)
axes[1].set_yticks(range(len(explained)), explained.index)
axes[1].invert_yaxis()
axes[1].set_xlabel('Share of the spread explained by one setting alone')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=8)
plt.tight_layout()
plt.show()
explained.round(3)

How the settings move `S_t` and `R_t`. Each square is the median over the seeds of a cell. All runs are used here, grokked or not.

In [ ]:
fig, axes = plt.subplots(2, len(S_VALUES), figsize=(3.3 * len(S_VALUES), 6.5), layout='constrained', squeeze=False)
for row, stat in enumerate(['S_t', 'log R_t']):
    for col, (ax, s) in enumerate(zip(axes[row], S_VALUES)):
        grid = everything[everything[KNOB] == s].pivot_table(index='width', columns='eta_kappa', values=stat, aggfunc='median')
        im = ax.imshow(grid, cmap='Greys', aspect='auto', vmin=everything[stat].quantile(0.02), vmax=everything[stat].quantile(0.98))
        ax.set_xticks(range(grid.shape[1]), [f'{w:g}' for w in grid.columns], rotation=45)
        ax.set_yticks(range(grid.shape[0]), grid.index)
        ax.set_title(f'{stat}, s = {s:g}')
        ax.grid(False)
        if col == 0:
            ax.set_ylabel('width')
    fig.colorbar(im, ax=axes[row], label=stat)
for ax in axes[1]:
    ax.set_xlabel('weight decay ηλ')
plt.show()

Last, do the statistics follow the seed noise? We subtract each cell's average from every run, for $\log T$ and for each statistic, and plot what is left. A clear slope would mean the kernel tells seeds of the same cell apart.

In [ ]:
within_T = log_T - log_T.groupby(data['cell']).transform('mean')
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, stat in zip(axes, covs):
    within = data[stat] - data[stat].groupby(data['cell']).transform('mean')
    ax.scatter(within, within_T, s=10, color=KERNEL_COLOURS[stat], alpha=0.6)
    ax.set_title(f'{stat}:  r = {np.corrcoef(within, within_T)[0, 1]:+.2f}')
    ax.set_xlabel(f'{stat} minus its cell average')
axes[0].set_ylabel('log T minus its cell average')
plt.tight_layout()
plt.show()

The share of the spread in $\log T$ that lies between cells is the ceiling on $R^2$ for any model of these runs, because a model that knew each cell's average would reach it. The rest of this notebook compares each model with that ceiling.

**Takeaway.** To be written once the MNIST sweep has run.

## Part 3. What kind of curve? A ladder of models

**Question.** Is the relation a straight line? If not, what is the simplest model that captures it?

Each step up the ladder allows one more kind of shape. With $z_A$, $z_S$, $z_R$ the standardised statistics:
1. **Linear, raw `R_t`.** $\beta_0 + \beta_A z_A + \beta_S z_S + \beta_R z_R$ with `R_t` on its own scale.
2. **Linear.** The same with log `R_t` in place of `R_t`. From here on, $z_R$ is the standardised log `R_t`.
3. **Linear + $S_t^2$.** Model 2 plus a square of `S_t`, which lets the `S_t` curve rise and fall.
4. **Additive splines.** $\beta_0 + f_A(z_A) + f_S(z_S) + f_R(z_R)$, each $f$ a cubic spline with 5 knots. Any smooth bend, but no interactions.
5. **Splines + products.** Model 4 plus $\gamma_{AS} z_A z_S + \gamma_{AR} z_A z_R + \gamma_{SR} z_S z_R$.
6. **Kernel ridge regression** with an RBF kernel. Any smooth surface, with interactions of every order.

If a step beats the one before it, the shape it adds is real. Kernel ridge has no intercept, so far from the data its prediction falls to 0. We therefore subtract the mean of $\log T$ before fitting and add it back after.

In [ ]:
data['S_t²'] = data['S_t'] ** 2
linear = lambda: make_pipeline(StandardScaler(), LinearRegression())
splines = lambda: make_pipeline(StandardScaler(), SplineTransformer(n_knots=5, degree=3), RidgeCV(np.logspace(-3, 3, 13)))
splines_products = lambda: make_pipeline(StandardScaler(), make_union(SplineTransformer(n_knots=5, degree=3),
                                                                      PolynomialFeatures(2, interaction_only=True, include_bias=False)),
                                         RidgeCV(np.logspace(-3, 3, 13)))
kernel_ridge = lambda: TransformedTargetRegressor(
    GridSearchCV(make_pipeline(StandardScaler(), KernelRidge(kernel='rbf')),
                 {'kernelridge__alpha': [0.01, 0.1, 1], 'kernelridge__gamma': [0.1, 0.5, 2]}, cv=GroupKFold(5)),
    transformer=StandardScaler())
ladder = {     # name: (model, covariates)
    'linear, raw R_t': (linear, kernel),
    'linear': (linear, covs),
    'linear + S_t²': (linear, ['A_t', 'S_t', 'S_t²', 'log R_t']),
    'additive splines': (splines, covs),
    'splines + products': (splines_products, covs),
    'kernel ridge': (kernel_ridge, covs),
}

pred = {}
for hide in ['cell', KNOB]:
    for name, (make, cols) in ladder.items():
        p = pd.Series(index=data.index, dtype=float)
        for group in data[hide].unique():
            train, test = data[hide] != group, data[hide] == group
            extra = {'groups': data.loc[train, 'cell']} if name == 'kernel ridge' else {}
            p[test] = make().fit(data.loc[train, cols], log_T[train], **extra).predict(data.loc[test, cols])
        pred[hide, name] = p
LADDER_COLOURS = [PALE, GREY, KERNEL_COLOURS['S_t'], INK, INK, KERNEL_COLOURS['A_t']]

### Hold out one cell

In [ ]:
scores = pd.DataFrame({name: {'R²': r2_score(log_T, pred['cell', name]), 'concordance': concordance_index(log_T, pred['cell', name])}
                       for name in ladder}).T

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, col in zip(axes, scores):
    ax.barh(scores.index, scores[col], color=LADDER_COLOURS)
    ax.set_xlabel(col + ' on held-out cells')
axes[0].invert_yaxis()
axes[1].set_xlim(0.5, 1.0)
plt.tight_layout()
plt.show()
scores.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### Hold out one s

We score each held-out s on its own. The table also gives $R^2$ over all folds, which asks whether the model gets the actual step right. A good ranking does not need that.

In [ ]:
S_LABELS = [f's = {s:g}' for s in sorted(data[KNOB].unique())]
rows = {}
for name in ladder:
    p = pred[KNOB, name]
    rows[name] = {f's = {s:g}': concordance_index(log_T[data[KNOB] == s], p[data[KNOB] == s]) for s in sorted(data[KNOB].unique())}
    rows[name]['mean concordance'] = np.mean(list(rows[name].values()))
    rows[name]['R²'] = r2_score(log_T, p)
s_scores = pd.DataFrame(rows).T

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(s_scores.index, s_scores['mean concordance'], color=LADDER_COLOURS)
for i, (name, row) in enumerate(s_scores.iterrows()):
    ax.scatter(row[S_LABELS], [i] * len(S_LABELS), color='white', edgecolors=INK, s=25, zorder=3)
ax.axvline(0.5, color=GREY, linestyle=':')
ax.set_xlim(min(0.4, s_scores[S_LABELS].min().min() - 0.02), max(0.8, s_scores[S_LABELS].max().max() + 0.02))
ax.invert_yaxis()
ax.set_xlabel('Concordance on the held-out s (bar = mean, dots = each s)')
plt.tight_layout()
plt.show()
s_scores.round(3)

Why can $R^2$ fall below 0 while the ranking stays good? Here are the predictions against the truth. A model can get the order right and still shift every run of a held-out s up or down.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, name in zip(axes, ['additive splines', 'kernel ridge']):
    for s in sorted(data[KNOB].unique()):
        m = data[KNOB] == s
        ax.scatter(log_T[m], pred[KNOB, name][m], s=10, color=S_COLOURS[s], alpha=0.6, label=f's = {s:g} held out')
    lims = [log_T.min(), log_T.max()]
    ax.plot(lims, lims, color=GREY, linestyle=':')
    ax.set_xlabel('True log T')
    ax.set_title(name)
axes[0].set_ylabel('Predicted log T')
axes[1].legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 4. What shape is each curve?

**Question.** For each statistic, how does the predicted grokking step change as it grows?

We fit kernel ridge and the additive splines once on all grokked runs and draw each statistic's partial dependence: slide it from low to high, keep the other two at each run's own values, and average the predictions. Where the two models agree, the shape does not depend on the model.

In [ ]:
krr = kernel_ridge().fit(data[covs], log_T, groups=data['cell'])
spl = splines().fit(data[covs], log_T)
print('kernel ridge settings:', krr.regressor_.best_params_)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, stat in zip(axes, covs):
    grid = np.linspace(data[stat].quantile(0.02), data[stat].quantile(0.98), 50)
    for model, style in [(krr, dict(color=KERNEL_COLOURS[stat], linewidth=2.5, label='kernel ridge')),
                         (spl, dict(color=INK, linestyle='--', label='additive splines'))]:
        curve = []
        for v in grid:
            X = data[covs].copy()
            X[stat] = v
            curve.append(model.predict(X).mean())
        ax.plot(np.exp(grid) if stat == 'log R_t' else grid, np.exp(curve), **style)
    ax.set_yscale('log')
    ax.set_xlabel(stat.replace('log ', ''))
axes[2].set_xscale('log')
axes[2].set_xlabel('R_t (log scale)')
axes[0].set_ylabel('Predicted grokking step')
axes[0].legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

Where only one value of s has runs, the shape of a curve rests on that value alone. Check the range of each statistic for each s in Part 1 before reading a bend.

**Takeaway.** To be written once the MNIST sweep has run.

## Part 5. Which statistic carries the most information?

**Question.** If we remove one statistic, can the other two make up for it?

We refit kernel ridge without each statistic in turn and score it on held-out cells. A large drop in $R^2$ means the other two cannot replace the missing one.

In [ ]:
dropped = {'all three': covs, 'without A_t': ['S_t', 'log R_t'], 'without S_t': ['A_t', 'log R_t'], 'without R_t': ['A_t', 'S_t']}
rows = {}
for name, cols in dropped.items():
    p = pd.Series(index=data.index, dtype=float)
    for group in data['cell'].unique():
        train, test = data['cell'] != group, data['cell'] == group
        p[test] = kernel_ridge().fit(data.loc[train, cols], log_T[train], groups=data.loc[train, 'cell']).predict(data.loc[test, cols])
    rows[name] = {'R² on held-out cells': r2_score(log_T, p), 'concordance': concordance_index(log_T, p)}
dropped = pd.DataFrame(rows).T

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(dropped.index, dropped['R² on held-out cells'], color=[INK] + [KERNEL_COLOURS[k] for k in kernel])
ax.invert_yaxis()
ax.set_xlabel('R² on held-out cells')
plt.tight_layout()
plt.show()
dropped.round(3)

### Two statistics at once

The map shows the kernel ridge prediction as `S_t` and `R_t` change together, with `A_t` at its median. The dots are the runs. Away from the dots the map is a guess.

In [ ]:
S, logR = np.meshgrid(np.linspace(data['S_t'].min(), data['S_t'].max(), 60), np.linspace(data['log R_t'].min(), data['log R_t'].max(), 60))
probe = pd.DataFrame({'A_t': data['A_t'].median(), 'S_t': S.ravel(), 'log R_t': logR.ravel()})
surface = krr.predict(probe).reshape(S.shape) / np.log(10)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
im = ax.contourf(S, np.exp(logR), surface, levels=12, cmap='Greys')
for s, g in data.groupby(KNOB):
    ax.scatter(g['S_t'], g['R_t'], s=8, color=S_COLOURS[s], alpha=0.7, label=f's = {s:g}')
fig.colorbar(im, label='log10 of predicted grokking step')
ax.set_yscale('log')
ax.set_xlabel('S_t')
ax.set_ylabel('R_t (log scale)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(False)
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 6. Do the statistics interact? It depends on the question

**Question.** Part 3 asks about grokked runs only. Notebook 3 also counts runs that never grok. Do interactions matter for both?

We ask three questions and answer each with two model families:

| Question | Runs | Score |
| --- | --- | --- |
| Does the run grok at all? | all runs | AUC (0.5 = coin toss) |
| When does it grok, counting "never" as "after the budget of 500,000 steps"? | all runs | concordance |
| When does a grokked run grok? | the runs that grokked | concordance |

Each family comes in two versions that differ only in whether the statistics may interact:
- **Splines.** One curve per statistic, or the same curves plus the three products $z_A z_S$, $z_A z_R$, $z_S z_R$. The model is a logistic regression for the first question, an AFT for the second, and a regression on $\log T$ for the third.
- **XGBoost.** Trees of depth 1 (each tree splits on one statistic, so no interactions) or depth 3 (a tree can split on two or three in a row).

The gain is the score with interactions minus the score without. The cell first prints in how many cells some seeds grok and others do not. If that number is small, whether a run groks is almost a property of its cell.

The first question needs both grokked and not grokked runs in every held-out s. A held-out s where every run groks, or none does, has no AUC and is left out of the mean.

In [ ]:
everything['grokked'] = everything['T'].notna().astype(int)
everything['T_or_budget'] = everything['T'].fillna(everything['last_step'])
mixed = everything.groupby('cell')['grokked'].agg(lambda g: 0 < g.mean() < 1)
print(f'{mixed.sum()} of {len(mixed)} cells have some seeds that grok and some that do not.')


def auc(y, p):
    # AUC needs both outcomes among the scored runs.
    return roc_auc_score(y, p) if y.nunique() == 2 else np.nan


rows = []
for question in ['grok at all?', 'when? (all runs)', 'when? (grokked only)']:
    runs_used = data if question == 'when? (grokked only)' else everything
    T_used = runs_used['T_or_budget'] if 'T_or_budget' in runs_used else runs_used['T']
    event = runs_used['grokked'] if 'grokked' in runs_used else pd.Series(1, index=runs_used.index)
    for family in ['splines', 'XGBoost']:
        for shape in ['additive', 'interactions']:
            for hide in ['cell', KNOB]:
                p = pd.Series(index=runs_used.index, dtype=float)
                for group in runs_used[hide].unique():
                    train, test = runs_used[hide] != group, runs_used[hide] == group
                    z = (runs_used[covs] - runs_used.loc[train, covs].mean()) / runs_used.loc[train, covs].std()

                    if family == 'splines':
                        basis = SplineTransformer(n_knots=5, degree=3, include_bias=False).fit(z[train])
                        X = pd.DataFrame(basis.transform(z), index=z.index).add_prefix('curve ')
                        if shape == 'interactions':
                            for a, b in cov_pairs:
                                X[f'{a} × {b}'] = z[a] * z[b]
                        if question == 'grok at all?':
                            p[test] = LogisticRegression(max_iter=5000).fit(X[train], event[train]).predict_proba(X[test])[:, 1]
                        elif question == 'when? (all runs)':
                            aft = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=T_used, grokked=event), 'T', 'grokked')
                            p[test] = aft.predict_median(X[test]).to_numpy().ravel()
                        else:
                            p[test] = RidgeCV(np.logspace(-3, 3, 13)).fit(X[train], np.log(T_used[train])).predict(X[test])
                    else:
                        params = {'max_depth': 1 if shape == 'additive' else 3, 'eta': 0.05, 'min_child_weight': 5, 'subsample': 0.8, 'seed': 0}
                        if question == 'grok at all?':
                            m = xgb.DMatrix(z[train], label=event[train])
                            params['objective'] = 'binary:logistic'
                        else:
                            m = xgb.DMatrix(z[train])
                            m.set_float_info('label_lower_bound', T_used[train])
                            m.set_float_info('label_upper_bound', np.where(event[train] == 1, T_used[train], np.inf))
                            params.update({'objective': 'survival:aft', 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 1.0})
                        p[test] = xgb.train(params, m, num_boost_round=300).predict(xgb.DMatrix(z[test]))

                # Score all runs together when holding out cells, and as the mean over the values of s when holding out s.
                parts = [runs_used.index] if hide == 'cell' else [runs_used.index[runs_used[KNOB] == s] for s in sorted(runs_used[KNOB].unique())]
                if question == 'grok at all?':
                    score = np.nanmean([auc(event[i], p[i]) for i in parts])
                else:
                    score = np.mean([concordance_index(T_used[i], p[i], event[i]) for i in parts])
                rows.append({'question': question, 'family': family, 'shape': shape, 'hide': hide, 'score': score})

inter = pd.DataFrame(rows).pivot_table(index=['question', 'family'], columns=['hide', 'shape'], values='score', sort=False, dropna=False)
for hide in ['cell', KNOB]:
    inter[hide, 'gain'] = inter[hide, 'interactions'] - inter[hide, 'additive']
inter = inter[[(h, s) for h in ['cell', KNOB] for s in ['additive', 'interactions', 'gain']]]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
labels = [f'{q}\n{f}' for q, f in inter.index]
for ax, hide in zip(axes, ['cell', KNOB]):
    ax.barh(range(len(inter)), inter[hide, 'gain'], color=[INK if f == 'XGBoost' else GREY for _, f in inter.index])
    ax.axvline(0, color=GREY, linestyle=':')
    ax.set_xlabel(f'Gain from interactions, holding out one {"cell" if hide == "cell" else "s"}')
axes[0].set_yticks(range(len(inter)), labels)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()
inter.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 7. The history of the kernel, and the settings

**Question.** Among grokked runs, does the history of the statistics up to $t^*$ predict better than their value at $t^*$? And how do both compare with the settings themselves?

The model is XGBoost regression on $\log T$, with its settings chosen on a validation set (see How the models are tested). It receives five sets of covariates:
- the three statistics at $t^*$
- the first 5 kernel steps only, long before any feature learning (the cell prints them)
- the full history: all kernel steps up to $t^*$, three values at each
- the settings alone: s, width and weight decay
- the settings together with the full history

If the history does as well as the settings, it is mostly recording the settings. Weight decay enters on a log scale, and no decay is placed one decade below the smallest decay in the data. Each fold uses one seed, and the folds run in parallel.

In [ ]:
XGB_GRID = [{'max_depth': d, 'eta': e, 'min_child_weight': w} for d, e, w in itertools.product([1, 2, 3, 4], [0.03, 0.1], [1, 5, 10])]
before = kern[kern['step'] <= T_STAR].pivot(index='run_id', columns='step', values=kernel)
history = np.stack([before[k].loc[data['run_id']].to_numpy() for k in kernel], axis=-1)     # runs × kernel steps up to t* × 3 statistics
history_steps = before['A_t'].columns.to_numpy()
print(f'{len(history_steps)} kernel steps up to t*. The first 5 are steps {[int(t) for t in history_steps[:5]]}.')
no_decay = data['eta_kappa'][data['eta_kappa'] > 0].min() / 10                             # where eta_kappa = 0 sits on the log scale
settings = np.column_stack([np.log(data[KNOB]), np.log(data['width']), np.log10(data['eta_kappa'].where(data['eta_kappa'] > 0, no_decay))])
inputs = {'XGBoost, statistics at t*': history[:, -1, :], 'XGBoost, first 5 steps': history[:, :5, :].reshape(len(data), -1),
          'XGBoost, full history': history.reshape(len(data), -1), 'XGBoost, settings': settings,
          'XGBoost, settings + full history': np.hstack([settings, history.reshape(len(data), -1)])}
y, cells = log_T.to_numpy(), data['cell'].to_numpy()


def tuned_fold(X, train, test, seed=0):
    # Choose the settings on 20% of the training cells, refit on all training runs, and predict the test runs.
    train_cells = np.unique(cells[train])
    val = train & np.isin(cells, np.random.default_rng(seed).choice(train_cells, round(0.2 * len(train_cells)), replace=False))
    fit = train & ~val
    best = (np.inf, None, None)
    for setting in XGB_GRID:
        params = {'objective': 'reg:squarederror', 'subsample': 0.8, 'nthread': 1, 'seed': seed, **setting}
        booster = xgb.train(params, xgb.DMatrix(X[fit], label=y[fit]), 1000, evals=[(xgb.DMatrix(X[val], label=y[val]), 'validation')],
                            early_stopping_rounds=50, verbose_eval=False)
        if booster.best_score < best[0]:
            best = (booster.best_score, params, booster.best_iteration + 1)
    return test, xgb.train(best[1], xgb.DMatrix(X[train], label=y[train]), best[2]).predict(xgb.DMatrix(X[test]))


rows = {}
for name, X in inputs.items():
    rows[name] = {}
    for hide in ['cell', KNOB]:
        folds = [((data[hide] != g).to_numpy(), (data[hide] == g).to_numpy()) for g in data[hide].unique()]
        p = np.zeros(len(data))
        for test, fold_pred in Parallel(n_jobs=-1)(delayed(tuned_fold)(X, train, test) for train, test in folds):
            p[test] = fold_pred
        if hide == 'cell':
            rows[name].update({'R² on held-out cells': r2_score(y, p), 'concordance on held-out cells': concordance_index(y, p)})
        else:
            rows[name]['concordance on held-out s'] = np.mean([concordance_index(y[data[KNOB] == s], p[data[KNOB] == s]) for s in sorted(data[KNOB].unique())])

earlier = pd.DataFrame({'R² on held-out cells': scores['R²'], 'concordance on held-out cells': scores['concordance'],
                        'concordance on held-out s': s_scores['mean concordance']}).loc[['linear + S_t²', 'kernel ridge']]
xgb_table = pd.concat([earlier, pd.DataFrame(rows).T])

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(xgb_table.index, xgb_table['R² on held-out cells'], color=[KERNEL_COLOURS['S_t'], KERNEL_COLOURS['A_t'], GREY, GREY, GREY, PALE, PALE])
ax.axvline(1 - explained.loc['log T', 'seed noise'], color=INK, linestyle='--')
ax.text(1 - explained.loc['log T', 'seed noise'], -0.7, ' ceiling from seed noise', fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('R² on held-out cells')
plt.tight_layout()
plt.show()
xgb_table.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Summary

| Question | Answer |
| --- | --- |
| How much is noise? | To be written once the MNIST sweep has run. |
| What sets the statistics? | To be written once the MNIST sweep has run. |
| Is the relation a straight line? | To be written once the MNIST sweep has run. |
| What shape? | To be written once the MNIST sweep has run. |
| Do the statistics interact? | To be written once the MNIST sweep has run. |
| Which statistic carries the most? | To be written once the MNIST sweep has run. |
| Kernel or settings? | To be written once the MNIST sweep has run. |

## How to model these data

To be written once the MNIST sweep has run. Judge every model against two references: the ceiling on $R^2$ set by seed noise (Part 2), and a model of the settings alone (Part 7).

## Conclusion

**Takeaway.** To be written once the MNIST sweep has run.